# Indexing MS MARCO v1 Passage by OpenSearch for Dense Encoder Model

Dense-encodes each passage server-side via the remotely hosted
`intfloat/e5-large-v2` model (passage encoder) registered in OpenSearch.

Unlike the Robust04 notebook there is **no chunking stage**: MS MARCO passages
are already passage-sized (~56 words on average), so each document maps to a
single flat `knn_vector` — no nested fields.

- [msmarco-passage](https://ir-datasets.com/msmarco-passage.html)
- Prerequisite: corpus downloaded via [dataset/msmarco-v1-passage](../../dataset/msmarco-v1-passage/README.md)
- Prerequisite: [ml_model_registration.ipynb](ml_model_registration.ipynb) with
  [model_hosting/e5-large-v2.py](../../model_hosting/e5-large-v2.py) running on the model host

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Index a Corpus for DPR Model

Encoding runs on the remote model host, so no local GPU / sentence-transformers is needed here.

In [ ]:
import ir_datasets
dataset_name = "msmarco-passage"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "msmarco_v1_passage_dpr"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

### Dense Encoding (server-side ingest pipeline)

Bulk sends *raw* passages; the ingest pipeline runs a single `text_embedding`
processor that calls the remote `e5-large-v2` model on the `text` field and
writes a flat 1024-dim `knn_vector` to `text_embedding`. The model server
prepends the `passage: ` prefix itself (see `/embed/passages`).

In [ ]:
pipeline_id = "msmarco_v1_passage_dense"
model_id = "your-model-id"  # e5-large-v2 passage encoding model registered in OpenSearch

In [ ]:
def create_dense_pipeline(
    pipeline_id: str,
    model_id: str,
    source_field: str = "text",
    embedding_field: str = "text_embedding",
    batch_size: int = 64,
) -> dict:
    """
    Create (or update) an ingest pipeline that dense-encodes each passage,
    fully server-side. No chunking stage: MS MARCO passages are already
    passage-sized, so `source_field` is embedded directly to a single flat
    knn_vector in `embedding_field`.

    `batch_size` bundles that many passages into a single model call (batch
    ingestion), so the GPU processes them as one padded batch instead of one
    at a time.
    """
    body = {
        "description": "Dense-encode each MS MARCO passage",
        "processors": [
            {
                "text_embedding": {
                    "model_id": model_id,
                    "field_map": {source_field: embedding_field},
                    "batch_size": batch_size,
                }
            },
        ],
    }
    return client.ingest.put_pipeline(id=pipeline_id, body=body)

response = create_dense_pipeline(pipeline_id, model_id, batch_size=64)
pprint.pprint(response)

Bulk indexing

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      # Enable approximate k-NN so the knn_vector field builds a search graph.
      "knn": True
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "text": { "type": "text" },
        # Flat per-passage dense vector produced by the text_embedding processor.
        # e5-large-v2 emits 1024-dim vectors tuned for cosine similarity.
        "text_embedding": {
            "type": "knn_vector",
            "dimension": 1024,
            "space_type": "cosinesimil"
        },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

In [ ]:
def prepare_documents(dataset):
    """
    Yield raw bulk actions. MS MARCO passages are GenericDoc (doc_id + text
    only, no title) and already passage-sized (~56 words on average), so
    unlike the Robust04 notebooks there is no chunking stage and no
    MAX_DOC_CHARS cap.
    """
    for doc in dataset.docs_iter():
        yield {
            "_id": doc.doc_id,  # Unique identifier for the passage
            "_source": {
                "docid": doc.doc_id,
                "text": doc.text,
            }
        }

**Scale note:** 8.8M passages all pass through the GPU encoder. At a
throughput of ~1-2k passages/s this is a **multi-hour run** (roughly 2-4 h);
benchmark a small slice first (e.g. `itertools.islice(dataset.docs_iter(), 10_000)`)
to project the full duration before committing.

In [ ]:
from opensearchpy.helpers import streaming_bulk

# Batch inference is configured server-side in the pipeline's `text_embedding`
# processor (batch_size=64). OpenSearch 3.x removed the `_bulk?batch_size=` query
# param (2.x only), so we must NOT pass one here — it 400s the whole request.
# Total for the progress bar (docs_count is instant for msmarco-passage).
total = dataset.docs_count()   # 8,841,823

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=128,                # 2 x processor batch_size; bar advances every 128 passages
        request_timeout=300,
        max_retries=3,
        initial_backoff=2,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed passage
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

---
### (Optional) Re-index passages missing from the first pass

If a run was interrupted, diff the corpus against what's actually in the index
and re-index just the missing ids through the same pipeline.

In [ ]:
from opensearchpy.helpers import scan

# All ids actually in the index (_source disabled -> fast).
# 8.8M ids fit comfortably in memory (a few hundred MB), but the scan takes a while.
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

In [ ]:
# Re-index the passages identified as missing by the scan diff, printing every error.
from opensearchpy.helpers import streaming_bulk

def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        doc = docstore.get(doc_id)
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "text": doc.text},
        }

retry_ok, retry_failed = 0, []
with tqdm(total=len(missing), desc="Re-indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_missing(dataset, missing),
        index=index_name,
        pipeline=pipeline_id,
        chunk_size=500,
        request_timeout=300,
        raise_on_error=False,
        raise_on_exception=False,
    ):
        bar.update(1)
        retry_ok += ok
        if not ok:
            retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed[:10]:     # full error detail for the first failures
    pprint.pprint(item)
    print("-" * 80)